[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/multi_pav_b3d_resolvido.ipynb)

# Exercício resolvido: Cotas dos pavimentos no build123d
## Fernando Ferraz Rbeiro




### Cotas dos pavimentos

Abaixo temoso algoritmo das cotas dos pavimentos gerado em aulas anteriores

In [ ]:
# cota_inicial = float(input("Digite a cota inicial: "))
# pap = float(input("Digite a distância de piso a piso "))
# n_pav = int(input("Digite o número de pavimentos: "))

cota_inicial = 0
pap = 3
n_pav = 30


lista_pav = []
for i in range(n_pav + 1):
  cota_atual =   cota_inicial + (pap * i)
  cota_atual = round(cota_atual, 2)
  lista_pav.append(cota_atual)
  print(f"Cota do pavimento {i} = {cota_atual}")
print(lista_pav)

### Adapte esse algoritmo para a geração de pavimentos como prismas retangulares no build123d

#### Instalação dos pacotes

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install build123d
    !pip install cadquery-simpleviewerelse:
    print("Not running in Colab, skipping package installation.")

### Importação dos pacotes

In [ ]:
from build123d import *
from cadquery_simple_viewer import show

### criação de caixas no build123d

In [ ]:
box = Box(30, 40, 3)
show(box, z=0)

In [ ]:
box = Box(30, 40, 3).translate((0, 0, 3/2))
show(box, z=0)

In [ ]:
box = Box(30, 40, 3, align=(Align.CENTER, Align.CENTER, Align.MIN))
show(box, z=0)

### Duas soluções para o problema

#### Solução 01

In [ ]:
cota_inicial = 0
pap = 3
n_pav = 30



lista_pav = []
for i in range(n_pav + 1):
  cota_atual =   cota_inicial + (pap * i)
  cota_atual = round(cota_atual, 2)
  # criar caixa na cota atual
  box = Box(30, 40, pap, align=(Align.CENTER, Align.CENTER, Align.MIN)).translate((0, 0, cota_atual))
  lista_pav.append(box)
  # print(f"Cota do pavimento {i} = {cota_atual}")
print(lista_pav)

show(lista_pav)

#### Solução 02

In [ ]:
from build123d import *

cota_inicial = 0
pap = 3
n_pav = 30

lista_pav = []
for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + (pap * i), 2)

    pav = extrude(
        Plane(origin=(0, 0, cota_atual)) * Rectangle(30, 40),  # desenha o retângulo já na cota atual, não precisa usar translate
        amount=pap                                             # extrusão em Z pelo valor de pap
    )

    lista_pav.append(pav)
    # print(f"Cota do pavimento {i} = {cota_atual}")

show(lista_pav)

#### Variação com scale

In [ ]:
import numpy as np

cota_inicial = 0
pap = 3
n_pav = 30

smooth_factor = 1/(np.pi * 2)

lista_pav = []
for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + (pap * i), 2)
    scale_factor = np.abs(np.sin(i) * smooth_factor + 1)

    pav = extrude(
        Plane(origin=(0, 0, cota_atual)) * Rectangle(30 * scale_factor, 40 * scale_factor),
        amount=pap
    )

    lista_pav.append(pav)
    # print(f"Cota do pavimento {i} = {cota_atual:.2f} | scale = {scale_factor:.4f}")

show(lista_pav)

### Rotação incremental


#### Solução 01

In [ ]:
# parâmetros
cota_inicial = 0
pap = 3
n_pav = 30
rot_inc = 2.5

# code
lista_pav = []
for i in range(n_pav + 1):
  cota_atual =   cota_inicial + (pap * i)
  cota_atual = round(cota_atual, 2)

  box = Box(30, 40, pap, align=(Align.CENTER, Align.CENTER, Align.MIN)).translate((0, 0, cota_atual))

  box = box.rotate(Axis.Z, i * rot_inc)
  lista_pav.append(box)
  # print(f"Cota do pavimento {i} = {cota_atual}")

# show
show(lista_pav)

#### Solução 02

In [ ]:
# parâmetros
cota_inicial = 0
pap = 3
n_pav = 30
rot_inc = 2.5

# code
lista_pav = []
for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + (pap * i), 2)
    scale_factor = np.abs(np.sin(i) * smooth_factor + 1)

    pav = extrude(
        Plane(origin=(0, 0, cota_atual)) * Rectangle(30, 40),
        amount=pap
    )
    pav = pav.rotate(Axis.Z, i * rot_inc)

    lista_pav.append(pav)

# show
show(lista_pav)

#### Solução 02 com fator de escalonamento

In [ ]:
# parâmetros
cota_inicial = 0
pap = 3
n_pav = 30
rot_inc = 2.5

smooth_factor = 1/(np.pi * 2)


# code
lista_pav = []
for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + (pap * i), 2)
    scale_factor = np.abs(np.sin(i) * smooth_factor + 1)

    pav = extrude(
        Plane(origin=(0, 0, cota_atual)) * Rectangle(30 * scale_factor, 40 * scale_factor),
        amount=pap
    )
    pav = pav.rotate(Axis.Z, i * rot_inc)

    lista_pav.append(pav)

# show
show(lista_pav)

## Exportando o modelo

In [ ]:
# Agrupa todos os pavimentos em um Compound e exporta para STEP
assy = Compound(children=lista_pav)

export_step(assy, "output.step")

# Alternativa: unir (fundir) os sólidos com "+" antes de exportar,
# equivalente ao mode="fused" do cq.Assembly.export() do CadQuery
# fused = lista_pav[0]
# for pav in lista_pav[1:]:
#     fused = fused + pav
# export_step(fused, "output.step")